## **Tasa de Interes - Sistema Bancario**

In [106]:
## Librerias
import pandas as pd
import datetime as dt
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
import time

## formato
pd.options.display.max_columns = 200 ## mayor numero de columnas a visualizar


Se consultaran datos de la tasa usando la web de la SBS. 

In [44]:
web = "https://www.sbs.gob.pe/app/pp/EstadisticasSAEEPortal/Paginas/TIActivaTipoCreditoEmpresa.aspx?tip=B"

rango de fechas, los ultimos 6 meses. Considerar que la informacion solo esta disponible de lunes a viernes. 

In [131]:
## rango de fechas

fecha_inicio = "2026-01-05"
fecha_fin = dt.date.today()

fechas = pd.date_range(
    start=fecha_inicio,
    end=fecha_fin,
    freq="B"
)

fecha_consulta = []

for fecha in fechas:

    periodo = fecha.strftime("%Y%m")   # YYYYMM
    fecha_str = fecha.strftime("%d_%m_%Y")

    nombre_archivo = f"tasa_interes_{fecha_str}_{periodo}.xlsx"

    fecha_consulta.append({
        "periodo": periodo,
        "fecha": fecha_str,
        "nombre_archivo": nombre_archivo
    })

fecha_consulta[-1]

{'periodo': '202606',
 'fecha': '01_06_2026',
 'nombre_archivo': 'tasa_interes_01_06_2026_202606.xlsx'}

### **Descarga Informacion**

In [100]:
path_raw = rf"C:\Users\PC\Desktop\Proyectos\Proyectos_Py\9.Tasas_Interes\data\raw"

In [101]:
chrome_options = Options()
chrome_options.add_experimental_option("prefs", {
    "download.default_directory": path_raw,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 20)

url = "https://www.sbs.gob.pe/app/pp/EstadisticasSAEEPortal/Paginas/TIActivaTipoCreditoEmpresa.aspx?tip=B"
driver.get(url)

In [102]:
def consulta_tasa_interes(input_fecha: str):

    wait = WebDriverWait(driver, 10)

    # INGRESAR FECHA
    fecha_input = wait.until(
        EC.presence_of_element_located((By.ID, "ctl00_cphContent_rdpDate_dateInput"))
    )

    fecha_input.clear()
    fecha_input.send_keys(input_fecha)

    time.sleep(3)

    # CLICK CONSULTAR
    consultar_btn = driver.find_element(By.ID, "ctl00_cphContent_btnConsultar")
    consultar_btn.click()

    time.sleep(3)

    # CLICK EXPORTAR (esperar que sea clickeable)}
    try:
        exportar_btn = wait.until(
            EC.presence_of_element_located((By.ID, "ctl00_cphContent_btnExportar"))
        )
        exportar_btn.click()
        print(f"Descarga iniciada para {input_fecha}")
        return True

    except TimeoutException:
        print(f"No existe botón exportar para {input_fecha}")
        return False

In [103]:
def renombrar_ultimo_archivo(path_raw, nuevo_nombre):

    time.sleep(3) 

    archivos = []

    for f in os.listdir(path_raw):
        if f.endswith(".xlsx"):
            archivos.append(f)

    if not archivos:
        raise FileNotFoundError("No se encontró archivo descargado.")

    # tomar el archivo más reciente
    rutas = []

    for f in archivos:
        ruta = os.path.join(path_raw, f)
        rutas.append(ruta)
    
    archivo_reciente = max(rutas, key=os.path.getctime)

    nueva_ruta = os.path.join(path_raw, nuevo_nombre)

    os.rename(archivo_reciente, nueva_ruta)

    print(f"Archivo guardado como: {nuevo_nombre}")

In [ ]:
## en caso se requiera todos los archivos
for fecha_i in fecha_consulta[-1]:

    fecha = fecha_i['fecha']
    periodo = fecha_i['periodo']
    nombre_archivo = fecha_i['nombre_archivo']  # ejemplo: Tasa_201901.xls

    print(f"Descargada  - Fecha consulta: {fecha}")

    descarga_ok = consulta_tasa_interes(fecha)

    if descarga_ok:
        renombrar_ultimo_archivo(path_raw, nombre_archivo)

In [ ]:
## en caso se requiera solo el ultimo 
ultima_fecha = fecha_consulta[-1]

fecha = ultima_fecha['fecha']
periodo = ultima_fecha['periodo']
nombre_archivo = ultima_fecha['nombre_archivo']  # ejemplo: Tasa_201901.xls

print(f"Descargada  - Fecha consulta: {fecha}")

descarga_ok = consulta_tasa_interes(fecha)

if descarga_ok:
    renombrar_ultimo_archivo(path_raw, nombre_archivo)

Descargada  - Fecha consulta: 01_06_2026
Descarga iniciada para 01_06_2026
Archivo guardado como: tasa_interes_01_06_2026_202606.xlsx


### **Procesamiento de Datos**

In [111]:
fecha_consulta[81]

{'periodo': '202604',
 'fecha': '24_04_2026',
 'nombre_archivo': 'tasa_interes_24_04_2026_202604.xlsx'}

In [133]:
from pathlib import Path

In [132]:
path_raw = Path(r"C:\Users\PC\Desktop\Proyectos\Proyectos_Py\9.Tasas_Interes\data\raw")

In [137]:

data_tasas = []

for archivo in fecha_consulta:

    nombre_archivo = archivo["nombre_archivo"]

    path_file = path_raw / nombre_archivo

    try:

        data = pd.read_excel(
            path_file
            , skiprows = 6
            , skipfooter= 3
        )

        data.dropna(axis=1, how="all", inplace=True)
        data["Periodo"] = archivo["periodo"]
        data["Fecha"] = archivo["fecha"]


        data_tasas.append(data)
    
    except FileNotFoundError:
        print("Archivo No Encontrado", nombre_archivo)


Archivo No Encontrado tasa_interes_02_04_2026_202604.xlsx
Archivo No Encontrado tasa_interes_03_04_2026_202604.xlsx
Archivo No Encontrado tasa_interes_01_05_2026_202605.xlsx


In [140]:
data_tasas_final = []

for data_i in data_tasas:
    
    data_final_pv = pd.melt(data_i
                    , id_vars= ['Periodo', 'Fecha' , 'Tasa Anual (%)']
                    , var_name= "Bancos" 
                    , value_name="tasa_interes")
    
    data_tasas_final.append(data_final_pv)

In [157]:
data_tasas_final_df = pd.concat(data_tasas_final, ignore_index=True, axis = 0)

In [160]:
data_tasas_final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99536 entries, 0 to 99535
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Periodo         99536 non-null  object
 1   Fecha           99536 non-null  object
 2   Tasa Anual (%)  98024 non-null  object
 3   Bancos          99536 non-null  object
 4   tasa_interes    97520 non-null  object
 5   Tipo_Tasa       98024 non-null  object
dtypes: object(6)
memory usage: 4.6+ MB


In [163]:
data_tasas_final_df

,Periodo,Fecha,Tasa Anual (%),Bancos,tasa_interes,Tipo_Tasa
0,202601,05_01_2026,Corporativos,BBVA,5.19,Corporativos
1,202601,05_01_2026,Descuentos,BBVA,5.68,Descuentos
2,202601,05_01_2026,Préstamos hasta 30 días,BBVA,4.69,Préstamos hasta 30 días
3,202601,05_01_2026,Préstamos de 31 a 90 días,BBVA,5.18,Préstamos de 31 a 90 días
4,202601,05_01_2026,Préstamos de 91 a 180 días,BBVA,5.87,Préstamos de 91 a 180 días
...,...,...,...,...,...,...
99531,202606,01_06_2026,Préstamos no Revolventes para libre disp...,Promedio,81.93,Préstamos no Revolventes para libre disponibi...
99532,202606,01_06_2026,Préstamos no Revolventes para libre disp...,Promedio,24.21,Préstamos no Revolventes para libre disponibi...
99533,202606,01_06_2026,Créditos pignoraticios,Promedio,49.79,Créditos pignoraticios
99534,202606,01_06_2026,Hipotecarios,Promedio,7.86,Hipotecarios


In [ ]:
data_tasas_final_df["Tipo_Tasa"] = (data_tasas_final_df["Tasa Anual (%)"].str.strip())
data_tasas_final_df["tasa_interes"] = data_tasas_final_df["tasa_interes"] 

In [159]:
data_tasas_final_df

,Periodo,Fecha,Tasa Anual (%),Bancos,tasa_interes,Tipo_Tasa
0,202601,05_01_2026,Corporativos,BBVA,5.19,Corporativos
1,202601,05_01_2026,Descuentos,BBVA,5.68,Descuentos
2,202601,05_01_2026,Préstamos hasta 30 días,BBVA,4.69,Préstamos hasta 30 días
3,202601,05_01_2026,Préstamos de 31 a 90 días,BBVA,5.18,Préstamos de 31 a 90 días
4,202601,05_01_2026,Préstamos de 91 a 180 días,BBVA,5.87,Préstamos de 91 a 180 días
...,...,...,...,...,...,...
99531,202606,01_06_2026,Préstamos no Revolventes para libre disp...,Promedio,81.93,Préstamos no Revolventes para libre disponibi...
99532,202606,01_06_2026,Préstamos no Revolventes para libre disp...,Promedio,24.21,Préstamos no Revolventes para libre disponibi...
99533,202606,01_06_2026,Créditos pignoraticios,Promedio,49.79,Créditos pignoraticios
99534,202606,01_06_2026,Hipotecarios,Promedio,7.86,Hipotecarios


In [155]:
data_tasas_final_df["Tasa Anual (%)"].unique().tolist()

['Corporativos',
 'Descuentos',
 'Préstamos hasta 30 días',
 'Préstamos de 31 a 90 días',
 'Préstamos de 91 a 180 días',
 'Préstamos de 181 a 360 días',
 'Préstamos a más de 360 días',
 'Grandes Empresas',
 'Medianas Empresas',
 'Pequeñas Empresas',
 'Microempresas',
 'Tarjetas de Crédito',
 'Préstamos Revolventes',
 'Préstamos a cuota fija hasta 30 días',
 'Préstamos  a cuota fija de 31 a 90 días',
 'Préstamos  a cuota fija de 91 a 180 días',
 'Préstamos a cuota fija de 181 a 360 días',
 'Préstamos a cuota fija a más de 360 días',
 'Consumo',
 'Préstamos no  Revolventes para automóviles',
 'Préstamos no  Revolventes para libre disponibilidad hasta 360 días',
 'Préstamos no  Revolventes para libre disponibilidad a más de 360 días',
 'Créditos pignoraticios',
 'Hipotecarios',
 'Préstamos hipotecarios para vivienda',
 nan,
 'Nota: Cuadro elaborado sobre la base de la información remitida diariamente por las Empresas Bancarias a través del Reporte N°6. Las tasas de interés tienen carácter